In [1]:
import psycopg2
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import networkx as nx
import pandas as pd
import numpy as np
from pyvis.network import Network
from extraction.loader import load_cloudburst_signals

In [2]:
data = load_cloudburst_signals()

SSH tunnel connected
SSH tunnel connected


In [3]:
data

,pid,entity_id,signal_type,source_datetime,commodity,username,channel_participants
0,120246,-1001501522090,sell,2022-04-10 21:11:13,BRN,BrnTokenGlobal,NaN
1,120247,-1001501522090,sell,2022-04-10 21:11:51,BRN,BrnTokenGlobal,NaN
2,120249,-1001501522090,sell,2022-04-10 21:15:22,BRN,BrnTokenGlobal,NaN
3,120252,-1001501522090,sell,2022-04-10 21:24:38,BRN,BrnTokenGlobal,NaN
4,120253,-1001501522090,sell,2022-04-10 21:46:00,BRN,BrnTokenGlobal,NaN
...,...,...,...,...,...,...,...
172853,425911,-1001094499146,time,2023-01-08 17:00:47,BPS,Big_Pumps_Signals_Global,284183.0
172854,425910,-1001094499146,buy,2023-01-08 17:00:47,BPS,Big_Pumps_Signals_Global,284183.0
172855,411573,-1001376762273,buy,2022-12-25 15:12:04,MOON,crypto_pump_wenlambo,4912.0
172856,419608,-1001376762273,buy,2023-01-03 15:00:06,ZERO,crypto_pump_wenlambo,4912.0


In [15]:
#Round to signal_hour
data['signal_minute'] = pd.to_datetime(data['source_datetime']).dt.round(freq='min')#Pivot table for correlation
data_pivot = data.pivot_table(
    columns="entity_id",
    index=["signal_minute", "signal_type"], aggfunc='count', values = 'pid')

In [16]:
data_pivot

entity_id                        -1001003243820  -1001094499146  \
signal_minute       signal_type                                   
2017-09-22 06:29:00 sell                    NaN             NaN   
2017-09-22 17:50:00 sell                    NaN             NaN   
2017-09-26 03:45:00 sell                    NaN             NaN   
2017-09-26 03:47:00 sell                    NaN             NaN   
2017-09-26 11:01:00 sell                    NaN             NaN   
...                                         ...             ...   
2023-01-08 17:00:00 buy                     NaN             NaN   
2023-01-08 17:01:00 buy                     NaN             1.0   
                    time                    NaN             1.0   
2023-01-08 17:08:00 sell                    NaN             NaN   
2023-01-08 17:09:00 sell                    NaN             NaN   

entity_id                        -1001129771868  -1001130260284  \
signal_minute       signal_type                                   
2017-09-22 06:29:00 sell                    NaN             NaN   
2017-09-22 17:50:00 sell                    NaN             NaN   
2017-09-26 03:45:00 sell                    NaN             NaN   
2017-09-26 03:47:00 sell                    NaN             NaN   
2017-09-26 11:01:00 sell                    NaN             NaN   
...                                         ...             ...   
2023-01-08 17:00:00 buy                     NaN             NaN   
2023-01-08 17:01:00 buy                     NaN             NaN   
                    time                    NaN             NaN   
2023-01-08 17:08:00 sell                    NaN             NaN   
2023-01-08 17:09:00 sell                    NaN             NaN   

entity_id                        -1001133895703  -1001144292688  \
signal_minute       signal_type                                   
2017-09-22 06:29:00 sell                    1.0             NaN   
2017-09-22 17:50:00 sell                    1.0             NaN   
2017-09-26 03:45:00 sell                    1.0             NaN   
2017-09-26 03:47:00 sell                    1.0             NaN   
2017-09-26 11:01:00 sell                    1.0             NaN   
...                                         ...             ...   
2023-01-08 17:00:00 buy                     NaN             NaN   
2023-01-08 17:01:00 buy                     NaN             NaN   
                    time                    NaN             NaN   
2023-01-08 17:08:00 sell                    NaN             NaN   
2023-01-08 17:09:00 sell                    NaN             NaN   

entity_id                        -1001147810456  -1001148414624  \
signal_minute       signal_type                                   
2017-09-22 06:29:00 sell                    NaN             NaN   
2017-09-22 17:50:00 sell                    NaN             NaN   
2017-09-26 03:45:00 sell                    NaN             NaN   
2017-09-26 03:47:00 sell                    NaN             NaN   
2017-09-26 11:01:00 sell                    NaN             NaN   
...                                         ...             ...   
2023-01-08 17:00:00 buy                     NaN             NaN   
2023-01-08 17:01:00 buy                     NaN             NaN   
                    time                    NaN             NaN   
2023-01-08 17:08:00 sell                    NaN             NaN   
2023-01-08 17:09:00 sell                    NaN             NaN   

entity_id                        -1001155306123  -1001155361313  ...  \
signal_minute       signal_type                                  ...   
2017-09-22 06:29:00 sell                    NaN             NaN  ...   
2017-09-22 17:50:00 sell                    NaN             NaN  ...   
2017-09-26 03:45:00 sell                    NaN             NaN  ...   
2017-09-26 03:47:00 sell                    NaN             NaN  ...   
2017-09-26 11:01:00 sell                    NaN             NaN  ...   
...     

In [26]:
#
data_corr = data_pivot.corr()
data_corr_one_side = data_corr.where(np.triu(np.ones(data_corr.shape), k=1).astype(np.bool_))
entity_size = data_corr_one_side.sum(axis = 1).sort_values(ascending = False)
corr_melt = data_corr_one_side.stack()
col_names = ['channel1', 'channel2']
corr_melt.index.names = col_names
corr_melt = corr_melt.reset_index()
corr_melt.columns = col_names + ['weight']


In [18]:
data_corr

entity_id,-1001003243820,-1001094499146,-1001129771868,-1001130260284,-1001133895703,-1001144292688,-1001147810456,-1001148414624,-1001155306123,-1001155361313,...,-1001770033880,-1001773172908,-1001777267013,-1001777719021,-1001782609878,-1001788143320,-1001790519070,-1001790623382,-1001791613137,-1001887542800
entity_id,,,,,,,,,,,,,,,,,,,,,
-1001003243820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
-1001094499146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
-1001129771868,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
-1001130260284,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,-0.040522,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
-1001133895703,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
-1001788143320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
-1001790519070,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
-1001790623382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
corr_melt = corr_melt[corr_melt['weight'] > 0.2]
corr_melt

,channel1,channel2,weight
3,-1001144292688,-1001190942267,1.000000
4,-1001144292688,-1001342400398,1.000000
5,-1001155306123,-1001313911314,0.222222
7,-1001155306123,-1001547366843,0.632456
9,-1001155306123,-1001583172969,0.994850
...,...,...,...
204,-1001628956682,-1001758534036,0.486418
208,-1001662988814,-1001757689494,0.347481
209,-1001662988814,-1001758534036,0.311433
211,-1001674306258,-1001757689494,1.000000


In [37]:
g = nx.from_pandas_edgelist(corr_melt, source = col_names[0], target = col_names[1], edge_attr = True)
edge_widths = nx.get_edge_attributes(g, "weight")

entity_size_ordered = entity_size[[w for w in g.nodes()]]

In [38]:
#Label creation for ploting
labels = {}
for node in g.nodes():
    if node in entity_size.index:
        #set the node name as the key and the label as its value 
        node_name = data[data['entity_id'] == node]['username'].values
        if len(node_name) > 0:
            labels[node] =  node_name[0]
        else:
            labels[node] = node


In [39]:
list_labels = [x for x in labels.values()]

list_width = list(edge_widths.values())
list_width = [i*i for i in list_width]


In [40]:
import plotly.graph_objects as go

pos = nx.spring_layout(g, k=0.5, iterations=20)

edge_x=[]
edge_y=[]
edges_list = [x for x in g.edges]

for edge in edges_list:
    edge_x+=[[pos[edge[0]][0],pos[edge[1]][0]]]
    edge_y+=[[pos[edge[0]][1],pos[edge[1]][1]]]

# Create the graph using the Scatter trace type
data_trace = []

Xv=[pos[k][0] for k in pos.keys()]
Yv=[pos[k][1] for k in pos.keys()]  

data_trace.append(go.Scatter(
    x=Xv, y=Yv,
    mode='markers',
    hoverinfo='text',
    marker=dict(
        showscale=True,
        # colorscale options
        #'Greys' | 'YlGnBu' | 'Greens' | 'YlOrRd' | 'Bluered' | 'RdBu' |
        #'Reds' | 'Blues' | 'Picnic' | 'Rainbow' | 'Portland' | 'Jet' |
        #'Hot' | 'Blackbody' | 'Earth' | 'Electric' | 'Viridis' |
        colorscale='Bluered',
        reversescale=True,
        color=[],
        size=10,
        colorbar=dict(
            thickness=15,
            title='Node Connections',
            xanchor='left',
            titleside='right'
        ),
        line_width=2)))

node_adjacencies = []
node_text = []
for adjacencies, labels in zip(g.adjacency(),list_labels):
    node_adjacencies.append(len(adjacencies[1]))
    node_text.append(f"{labels} ({adjacencies[0]}) Connections: {len(adjacencies[1])}")

size_list = [float(i) for i in entity_size.values]
size_list = [i*(25/i**(1/1.4)) if i>0 else 0.1 for i in size_list]

data_trace[0].marker.color = node_adjacencies
data_trace[0].marker.size = size_list
data_trace[0].text = node_text

# Add the edges to the graph, each with its own weight
for i in range(len(edge_x)):
    data_trace.append(
        go.Scatter(
            x=edge_x[i],
            y=edge_y[i],
            mode="lines",
            name="Edges",
            opacity=0.5,
            line=dict(width=list_width[i]*10, color="grey",shape='spline'
        ))
    )

fig = go.Figure(data=data_trace,
             layout=go.Layout(
                title='''
                Telegram Pump & Dump Channels Correlation''',
                titlefont_size=19,
                showlegend=False,
                hovermode='closest',
                template="plotly_dark",
                margin=dict(b=20,l=5,r=5,t=50),
                annotations=[ dict(
                    text="(c) 2022 Cloudburst Technologies - All Rights Reserved",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002 ) ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )
fig.update_layout(
            title_text="Telegram Pump & Dump Channels Correlation", title_x=0.5
        )
fig.show()